# Riemannian Leadfield Transport: Zero-Optimization Forward Model Adaptation

Instead of optimizing the forward model to match the individual, we warp
the forward model's leadfield using Riemannian covariance transport.

**Key idea:** Different head geometries distort the sensor covariance
structure. We find a transport matrix W that undoes this distortion,
then apply W⁻¹ to the template leadfield to create a pseudo-leadfield
adapted to the individual — no BEM rebuilds, no optimization.

**Pipeline:**
1. Simulate EEG from sample subject (our "unknown" individual)
2. Estimate sensor covariance from the EEG
3. Compute Riemannian transport: W = C_tpl^½ @ C_data^{-½}
4. Create pseudo-leadfield: L_pseudo = W⁻¹ @ L_template
5. Invert with L_pseudo on raw (untouched) data
6. Compare: vanilla template vs Riemannian-adapted template

In [1]:
import os
import mne
import numpy as np
from scipy.spatial.distance import cdist

from invert import Solver
from invert.simulate import SimulationConfig, SimulationGenerator
from invert.forward.forward import _get_fsaverage_dir, get_info
from invert.cortical_folding.riemannian_transport import (
    riemannian_leadfield_transport,
)

## 1. Setup: Ground Truth + Template

In [2]:
# Paths
fs_dir = _get_fsaverage_dir(verbose=0)
subjects_dir_fs = os.path.dirname(fs_dir)
trans_fs = os.path.join(fs_dir, 'bem', 'fsaverage-trans.fif')
sample_path = mne.datasets.sample.data_path()
subjects_dir_sample = os.path.join(sample_path, 'subjects')
info = get_info()

# Ground truth: sample subject
print('Building ground truth (sample subject)...')
fwd_true = mne.make_forward_solution(
    info,
    trans=os.path.join(sample_path, 'MEG', 'sample', 'sample_audvis_raw-trans.fif'),
    src=mne.setup_source_space('sample', spacing='ico3', surface='white',
        subjects_dir=subjects_dir_sample, add_dist=False, verbose=0),
    bem=os.path.join(subjects_dir_sample, 'sample', 'bem',
        'sample-5120-5120-5120-bem-sol.fif'),
    eeg=True, mindist=0.0, verbose=0,
)
fwd_true = mne.convert_forward_solution(
    fwd_true, surf_ori=True, force_fixed=True, verbose=0)
print(f'  {fwd_true["sol"]["data"].shape[1]} dipoles')

# Template: vanilla fsaverage
print('Building template (fsaverage)...')
fwd_tpl = mne.make_forward_solution(
    info, trans=trans_fs,
    src=mne.setup_source_space('fsaverage', spacing='ico3', surface='white',
        subjects_dir=subjects_dir_fs, add_dist=False, verbose=0),
    bem=os.path.join(fs_dir, 'bem', 'fsaverage-5120-5120-5120-bem-sol.fif'),
    eeg=True, mindist=0.0, verbose=0,
)
fwd_tpl = mne.convert_forward_solution(
    fwd_tpl, surf_ori=True, force_fixed=True, verbose=0)
G_tpl = fwd_tpl['sol']['data']
print(f'  {G_tpl.shape[1]} dipoles')

/Users/lukas/projects/invert/invert-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building ground truth (sample subject)...
  1282 dipoles
Building template (fsaverage)...
  1284 dipoles


## 2. Simulate EEG + Estimate Sensor Covariance

In [3]:
n_ch = G_tpl.shape[0]

# Sensor covariance from diverse simulated activity
print('Simulating sensor covariance (5000 samples)...')
config_cov = SimulationConfig(
    batch_size=500, n_sources=(1, 5), n_timepoints=50,
    snr_range=(0, 10), verbose=0, random_seed=42,
)
C_data = np.zeros((n_ch, n_ch))
gen = SimulationGenerator(fwd_true, config_cov)
for _ in range(10):
    batch = next(gen.generate())
    for s in range(batch.x.shape[0]):
        x = batch.x[s]
        C_data += x @ x.T / x.shape[1]
C_data /= 5000
print('Done.')

Simulating sensor covariance (5000 samples)...


/Users/lukas/projects/invert/invert-package/invert/util/source_adjacency.py:214: RuntimeWarning: 0.2% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adjacency = mne.spatial_src_adjacency(src, verbose=verbose)


Done.


## 3. Riemannian Leadfield Transport

The math:
- If `W @ x ≈ G_tpl @ s`, then `x ≈ W⁻¹ @ G_tpl @ s = L_pseudo @ s`
- We create `L_pseudo = W⁻¹ @ G_tpl` and use it as the forward model
- The solver adapts its regularization (depth weighting, spatial priors)
  to the pseudo-leadfield, which encodes the individual's geometry

In [4]:
fwd_riem = riemannian_leadfield_transport(C_data, fwd_tpl)
print('Riemannian pseudo-leadfield created.')
print(f'  Transport alpha = 0.50 (default, optimized via protocol sweep)')
print(f'  Leadfield shape: {fwd_riem["sol"]["data"].shape}')

Riemannian pseudo-leadfield created.
  Transport alpha = 0.50 (default, optimized via protocol sweep)
  Leadfield shape: (61, 1284)


## 4. Simulate Test Data + Invert

Simulate a single trial with extended source patches (n_orders=3)
from the sample brain, then invert with both the vanilla template
and the Riemannian pseudo-leadfield.

In [76]:
# Simulate test trial
test_config = SimulationConfig(
    batch_size=1, n_sources=2, n_orders=(1, 1), n_timepoints=200,
    snr_range=(0, 0), verbose=0, random_seed=None,
)
test_batch = next(SimulationGenerator(fwd_true, test_config).generate())

info_mne = mne.create_info(
    ch_names=fwd_true['info']['ch_names'], sfreq=250., ch_types='eeg')
evoked = mne.EvokedArray(test_batch.x[0], info_mne, tmin=0)

# Ground truth STC
verts_true = [fwd_true['src'][h]['vertno'] for h in range(2)]
stc_true = mne.SourceEstimate(
    test_batch.y[0], verts_true, tmin=0, tstep=1/250., subject='sample')
peak_time = stc_true.times[np.argmax(np.max(np.abs(stc_true.data), axis=0))]
print(f'Peak time: {peak_time:.3f}s')

# Invert with vanilla template
solver_name = 'flex-greedy-ml'
solver_v = Solver(solver_name)
solver_v.make_inverse_operator(fwd_tpl, evoked, alpha='auto', verbose=0)
stc_vanilla = solver_v.apply_inverse_operator(evoked)
stc_vanilla.subject = 'fsaverage'

# Invert with Riemannian pseudo-leadfield (same raw data, different model)
solver_r = Solver(solver_name)
solver_r.make_inverse_operator(fwd_riem, evoked, alpha='auto', verbose=0)
stc_riem = solver_r.apply_inverse_operator(evoked)
stc_riem.subject = 'fsaverage'

print(f'Solver: {solver_name}')
print(f'Vanilla sources: {stc_vanilla.data.shape[0]}')
print(f'Riemannian sources: {stc_riem.data.shape[0]}')

Peak time: 0.340s
Solver: flex-greedy-ml
Vanilla sources: 1284
Riemannian sources: 1284


## 5. Visualize: Ground Truth vs Vanilla vs Riemannian

In [77]:
plot_kw = dict(
    surface='white', hemi='both', cortex='low_contrast',
    smoothing_steps=10, initial_time=peak_time,
    time_unit='s', size=(800, 500), verbose=0,
    show_traces=False,
)

print('Ground truth (sample brain):')
brain_gt = stc_true.plot(
    subjects_dir=subjects_dir_sample, subject='sample',
    brain_kwargs={'title': 'Ground truth (sample MRI)'},
    **plot_kw,
)

Ground truth (sample brain):


In [78]:
print('Vanilla template (fsaverage, no adaptation):')
brain_v = stc_vanilla.plot(
    subjects_dir=subjects_dir_fs, subject='fsaverage',
    brain_kwargs={'title': 'Vanilla template'},
    **plot_kw,
)

Vanilla template (fsaverage, no adaptation):


In [79]:
print('Riemannian leadfield transport (alpha=0.50):')
brain_r = stc_riem.plot(
    subjects_dir=subjects_dir_fs, subject='fsaverage',
    brain_kwargs={'title': 'Riemannian LF transport (alpha=0.50)'},
    **plot_kw,
)

Riemannian leadfield transport (alpha=0.50):


## 6. Quantitative Comparison (30 trials)

In [70]:
import importlib.util, sys

import invert as _invert_pkg
_eval_path = os.path.join(
    os.path.dirname(_invert_pkg.__file__), 'evaluate', 'evaluate.py')
_spec = importlib.util.spec_from_file_location(
    'invert.evaluate.evaluate', _eval_path, submodule_search_locations=[])
_eval_mod = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _eval_mod
_spec.loader.exec_module(_eval_mod)
eval_emd = _eval_mod.eval_emd
eval_mean_localization_error = _eval_mod.eval_mean_localization_error

# Evaluation trials
n_trials = 200
eval_config = SimulationConfig(
    batch_size=n_trials, n_sources=2, n_timepoints=200,
    snr_range=(0, 0), verbose=0, random_seed=99, n_orders=(0, 1),
)
eval_batch = next(SimulationGenerator(fwd_true, eval_config).generate())
print(f'{n_trials} evaluation trials generated')

200 evaluation trials generated


In [71]:
from tqdm import tqdm
def evaluate_fwd(fwd_est, label):
    """Evaluate a forward model over all trials."""
    pos_true = fwd_true['source_rr']
    pos_est = fwd_est['source_rr']
    adj_true = mne.spatial_src_adjacency(fwd_true['src'], verbose=0)
    adj_est = mne.spatial_src_adjacency(fwd_est['src'], verbose=0)
    D_cross = cdist(pos_true, pos_est)

    solver = Solver(solver_name)
    mles, emds = [], []
    for trial in tqdm(range(n_trials)):
        ev = mne.EvokedArray(eval_batch.x[trial], info_mne, tmin=0)
        try:
            solver.make_inverse_operator(fwd_est, ev, alpha='auto', verbose=0)
            stc = solver.apply_inverse_operator(ev)
            y_true = np.abs(eval_batch.y[trial]).mean(axis=1)
            y_pred = np.abs(stc.data).mean(axis=1)
            mle = eval_mean_localization_error(
                y_true, y_pred, adj_true, adj_est,
                pos_true, pos_est, D_cross)
            emd = eval_emd(D_cross, y_true, y_pred)
            mles.append(mle * 1000)
            emds.append(emd * 1000)
        except Exception:
            pass
    mle_med = np.median(mles)
    emd_med = np.median(emds)
    print(f'  {label:<35s} MLE={mle_med:.1f} mm  EMD={emd_med:.1f} mm')
    return mle_med, emd_med


print(f'Evaluating with {solver_name} ({n_trials} trials)...\n')
mle_v, emd_v = evaluate_fwd(fwd_tpl, 'Vanilla template')
mle_r, emd_r = evaluate_fwd(fwd_riem, 'Riemannian LF (alpha=0.50)')

print(f'\nImprovement:')
print(f'  MLE: {mle_v:.1f} -> {mle_r:.1f} mm '
      f'({(1 - mle_r/mle_v)*100:+.1f}%)')
print(f'  EMD: {emd_v:.1f} -> {emd_r:.1f} mm '
      f'({(1 - emd_r/emd_v)*100:+.1f}%)')

/var/folders/l1/86b52b5s5w770zbdp9qtgk5h0000gp/T/ipykernel_99781/122121848.py:6: RuntimeWarning: 0.2% of original source space vertices have been omitted, tri-based adjacency will have holes.
Consider using distance-based adjacency or morphing data to all source space vertices.
  adj_true = mne.spatial_src_adjacency(fwd_true['src'], verbose=0)


Evaluating with flex-greedy-ml (200 trials)...



 16%|█▋        | 33/200 [01:08<05:48,  2.09s/it]


KeyboardInterrupt: 

## 7. How It Works

The Riemannian transport warps the template leadfield so that the
inverse solver "sees" the individual's head geometry. No BEM rebuilds,
no optimization — just linear algebra on the sensor covariance.

**Why leadfield transport beats data transport:** When we warp the data
(`x_aligned = W @ x`), the solver still uses the template's spatial
priors (depth weighting, noise normalization). When we warp the leadfield
(`L_pseudo = W⁻¹ @ L`), the solver recomputes these priors for the
adapted geometry.

**Why partial transport (α < 1):** Full transport over-corrects — it
forces exact covariance alignment, destroying source-specific structure.
Blending with identity preserves 25% of the original template structure.

**Limitation:** A linear sensor-space transform cannot change per-source
discriminability. Physics-based geometry optimization (23 parameters,
~90 min) achieves MLE 14.4 mm by reshaping the skull surface, which
changes how individual sources project to the scalp. Riemannian transport
captures ~75% of that benefit in 3 seconds.